In [1]:
import numpy as np
from sklearn.decomposition import PCA

In [2]:
X = np.random.rand(1000,3)
print(X[:5])

[[0.11637338 0.41985336 0.21225104]
 [0.62484775 0.47036798 0.99158043]
 [0.76903725 0.39606392 0.21298679]
 [0.76155945 0.59194631 0.34031044]
 [0.31430642 0.43000063 0.09984469]]


In [4]:
pca = PCA(n_components=2)

X2D = pca.fit_transform(X)
print(X2D[:5])

[[-0.47242836 -0.15071912]
 [ 0.41056633  0.02174117]
 [ 0.01834307 -0.13611212]
 [ 0.0720113   0.0727528 ]
 [-0.3993481  -0.14247889]]


In [5]:
X3D_inv = pca.inverse_transform(X2D)
print(np.allclose(X3D_inv, X))

False


In [7]:
print(pca.explained_variance_ratio_)
print(pca.explained_variance_ratio_.sum())

[0.3430007  0.34053597]
0.6835366728849297


In [8]:
print("Original shape:", X.shape)
print("PCA shape:", X2D.shape)

Original shape: (1000, 3)
PCA shape: (1000, 2)


In [11]:
import pandas as pd 
df = pd.read_csv("../kapitel-3(frågor&koduppgift)/car_price_dataset.csv", sep=";")
print(df.head())
print(df.shape)

        Brand   Model  Year  Engine_Size Fuel_Type    Transmission  Mileage  \
0         Kia     Rio  2020          4.2    Diesel          Manual   289944   
1   Chevrolet  Malibu  2012          2.0    Hybrid       Automatic     5356   
2    Mercedes     GLA  2020          4.2    Diesel       Automatic   231440   
3        Audi      Q5  2023          2.0  Electric          Manual   160971   
4  Volkswagen    Golf  2003          2.6    Hybrid  Semi-Automatic   286618   

   Doors  Owner_Count  Price  
0      3            5   8501  
1      2            3  12092  
2      4            2  11171  
3      2            1  11780  
4      3            3   2867  
(10000, 10)


In [13]:
print(df.info())
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Brand         10000 non-null  str    
 1   Model         10000 non-null  str    
 2   Year          10000 non-null  int64  
 3   Engine_Size   10000 non-null  float64
 4   Fuel_Type     10000 non-null  str    
 5   Transmission  10000 non-null  str    
 6   Mileage       10000 non-null  int64  
 7   Doors         10000 non-null  int64  
 8   Owner_Count   10000 non-null  int64  
 9   Price         10000 non-null  int64  
dtypes: float64(1), int64(5), str(4)
memory usage: 1.0 MB
None
Brand           0
Model           0
Year            0
Engine_Size     0
Fuel_Type       0
Transmission    0
Mileage         0
Doors           0
Owner_Count     0
Price           0
dtype: int64


In [14]:
X = df.drop("Price", axis=1)
y = df["Price"]

In [15]:
print(X.shape)
print(y.shape)

(10000, 9)
(10000,)


In [16]:
X = pd.get_dummies(X, drop_first=True)
print(X.shape)

(10000, 48)


In [17]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
  X,y, test_size=0.2, random_state=42
)

In [18]:
print(X_train.shape)
print(X_test.shape)

(8000, 48)
(2000, 48)


In [19]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [20]:
from sklearn.decomposition import PCA 
pca = PCA(n_components=10)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)


In [21]:
print(X_train_pca.shape)
print(X_test_pca.shape)

(8000, 10)
(2000, 10)


In [22]:
print(pca.explained_variance_ratio_)
print(pca.explained_variance_ratio_.sum())

[0.044893   0.04486156 0.04479504 0.04475262 0.04467924 0.044631
 0.04455506 0.04095684 0.03153254 0.0280542 ]
0.41371111638893077


In [23]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np 
model = LinearRegression()
model.fit(X_train_pca, y_train)
y_pred = model.predict(X_test_pca)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print("RMSE:", rmse)
print("R2:", r2)

RMSE: 2982.4115550374577
R2: 0.031893683710872756


In [24]:
model_without_pca = LinearRegression()
model_without_pca.fit(X_train_scaled, y_train)
y_pred_without_pca = model_without_pca.predict(X_test_scaled)
rmse_without_pca = np.sqrt(mean_squared_error(y_test, y_pred_without_pca))
r2_without_pca = r2_score(y_test, y_pred_without_pca)
print("RMSE without PCA:", rmse_without_pca)
print("R2 without PCA:", r2_without_pca)

RMSE without PCA: 64.91473251816437
R2 without PCA: 0.999541357336514


In [25]:
print("With PCA")
print("RMSE:", rmse)
print("R2:", r2)
print("without PCA")
print("RMSE:", rmse_without_pca)
print("R2:", r2_without_pca)

With PCA
RMSE: 2982.4115550374577
R2: 0.031893683710872756
without PCA
RMSE: 64.91473251816437
R2: 0.999541357336514


Enligt resultatet som jag fick, så hade PCA minskat från 48 till 10 kolumner. Modellen fungerade mindre bra med PCA, eftersom värdet på RMSE var högre,medans R2 värdet var lägre. Detta data visar att modellen har förlorat viktigt info efter att antalet komponenterna minskades. Så testat utan PCA fungerade bättre i mitt test. 